In [12]:
! pip install imbalanced-learn
! pip install xgboost lightgbm catboost

In [13]:
import kagglehub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight
import warnings
import os
from sklearn.impute import SimpleImputer
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
warnings.filterwarnings('ignore')

In [14]:
dataset_path = kagglehub.dataset_download('mdhossanr/financial-transactions-dataset-for-analysis')
file_path = os.path.join(dataset_path, 'Financial Transactions.csv')
df = pd.read_csv(file_path, encoding='utf-8', engine='python', encoding_errors='ignore')

df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['hour'] = df['Timestamp'].dt.hour
df['day_of_week'] = df['Timestamp'].dt.dayofweek
df['weekend'] = df['day_of_week'].apply(lambda x: 1 if x > 4 else 0)
df['month'] = df['Timestamp'].dt.month
df['year'] = df['Timestamp'].dt.year

# Признаки по счетам
df['avg_amount_per_account'] = df.groupby('AccountID')['TransactionAmount'].transform('mean')
df['std_amount_per_account'] = df.groupby('AccountID')['TransactionAmount'].transform('std')
std_safe = df['std_amount_per_account'].replace(0, 1)
df['amount_zscore'] = (df['TransactionAmount'] - df['avg_amount_per_account']) / std_safe
df['amount_zscore'] = df['amount_zscore'].fillna(0)

# Временные аномалии
df_sorted = df.sort_values(['AccountID', 'Timestamp'])
df_sorted['time_since_last'] = df_sorted.groupby('AccountID')['Timestamp'].diff().dt.total_seconds()
df['time_since_last'] = df_sorted['time_since_last'].fillna(999999)

# Целевая переменная
df['is_anomaly'] = ((df['amount_zscore'].abs() > 2.5) | (df['time_since_last'] < 300)).astype(int)

# Увеличение числа аномалий
non_anomaly_indices = df[df['is_anomaly'] == 0].index.tolist()
needed = 100 - df['is_anomaly'].sum()
np.random.seed(42)
new_anomaly_indices = np.random.choice(non_anomaly_indices, needed, replace=False)
df.loc[new_anomaly_indices, 'TransactionAmount'] = df.loc[new_anomaly_indices, 'TransactionAmount'] * 3
df.loc[new_anomaly_indices, 'is_anomaly'] = 1

# Пересчёт признаков
df['avg_amount_per_account'] = df.groupby('AccountID')['TransactionAmount'].transform('mean')
df['std_amount_per_account'] = df.groupby('AccountID')['TransactionAmount'].transform('std')
std_safe = df['std_amount_per_account'].replace(0, 1)
df['amount_zscore'] = (df['TransactionAmount'] - df['avg_amount_per_account']) / std_safe
df['amount_zscore'] = df['amount_zscore'].fillna(0)

# Новые признаки
# Доля суммы от баланса
df['balance_ratio'] = df['TransactionAmount'] / (df['AccountBalance'] + 1e-6)

# Крупная транзакция (порог 4000)
df['is_large'] = (df['TransactionAmount'] > 4000).astype(int)

# Циклическое кодирование часа
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Дни с начала истории
df['days_since_start'] = (df['Timestamp'] - df['Timestamp'].min()).dt.days

# Скользящее среднее суммы по счёту
df = df.sort_values(['AccountID', 'Timestamp'])
df['rolling_mean_amount'] = df.groupby('AccountID')['TransactionAmount'].transform(lambda x: x.rolling(5, min_periods=1).mean())

base_features = ['TransactionAmount', 'AccountBalance', 'hour', 'day_of_week', 'weekend', 'amount_zscore']
new_features = ['balance_ratio', 'is_large', 'hour_sin', 'hour_cos', 'days_since_start', 'rolling_mean_amount']
all_features = base_features + new_features

X = df[all_features]
y = df['is_anomaly']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# RandomForest с параметрами из baseline
rf_baseline = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    min_samples_leaf=2,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)

# Улучшенный RandomForest
rf_improved = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=1,
    min_samples_split=2,
    class_weight='balanced',
    random_state=42
)

# XGBoost
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# LightGBM
lgbm = LGBMClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    class_weight='balanced',
    random_state=42,
    verbose=-1
)

# CatBoost
catboost = CatBoostClassifier(
    iterations=100,
    depth=6,
    learning_rate=0.1,
    class_weights=[1, len(y_train[y_train==0])/len(y_train[y_train==1])],
    random_seed=42,
    verbose=False
)

# Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

models = {
    'Baseline RF (old features)': rf_baseline,
    'Improved RF (new features)': rf_improved,
    'XGBoost': xgb,
    'LightGBM': lgbm,
    'CatBoost': catboost,
    'GradientBoosting': gb
}

results = []
for name, model in models.items():
    if name in ['XGBoost', 'LightGBM', 'CatBoost', 'GradientBoosting']:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    results.append({
        'Model': name,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'Precision': precision,
        'Recall': recall
    })

    print(f'F1-Score: {f1:.4f}')
    print(f'ROC-AUC: {roc_auc:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(classification_report(y_test, y_pred, zero_division=0))

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

Using Colab cache for faster access to the 'financial-transactions-dataset-for-analysis' dataset.
F1-Score: 0.7879
ROC-AUC: 0.8429
Precision: 1.0000
Recall: 0.6500
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7464
           1       1.00      0.65      0.79        20

    accuracy                           1.00      7484
   macro avg       1.00      0.82      0.89      7484
weighted avg       1.00      1.00      1.00      7484

F1-Score: 0.7879
ROC-AUC: 0.7679
Precision: 1.0000
Recall: 0.6500
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7464
           1       1.00      0.65      0.79        20

    accuracy                           1.00      7484
   macro avg       1.00      0.82      0.89      7484
weighted avg       1.00      1.00      1.00      7484

F1-Score: 0.6842
ROC-AUC: 0.8300
Precision: 0.7222
Recall: 0.6500
              precision    recall  f1-score   supp

*На основе результатов я смогла выделить несколько важных проблем*:
1) Сильный дисбаланс классов
2) Все модели имеют recall 0.60-0.65, т.е. модели находят только 12-13 из 20 аномалий
3) CatBoost показывает аномально низкий precision (0.245)
4) GradientBoosting показывает лучший ROC-AUC (0.85), но F1 всего 0.65

***Выводы***:
1) В тестовой выборке всего 20 аномалий, что не несет в себе статистическую значимость
2) Модели не видят достаточно примеров аномалий для обучения
3) Нужно увеличить долю аномалий или использовать дополнительные методы

Для повышения качества и устранения дисбаланса я решила протестировать следующие приемы:
1) Увеличение доли аномалий до 1-2%
2) Использование SMOTE, ADASYN и SMOTETomek для дисбаланса классов
3) Поиск оптимального порога классификации по F1-score

In [15]:
df = pd.read_csv(file_path, encoding='utf-8', engine='python', encoding_errors='ignore')

df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['hour'] = df['Timestamp'].dt.hour
df['day_of_week'] = df['Timestamp'].dt.dayofweek
df['weekend'] = df['day_of_week'].apply(lambda x: 1 if x > 4 else 0)
df['month'] = df['Timestamp'].dt.month
df['year'] = df['Timestamp'].dt.year

df['avg_amount_per_account'] = df.groupby('AccountID')['TransactionAmount'].transform('mean')
df['std_amount_per_account'] = df.groupby('AccountID')['TransactionAmount'].transform('std')
std_safe = df['std_amount_per_account'].replace(0, 1)
df['amount_zscore'] = (df['TransactionAmount'] - df['avg_amount_per_account']) / std_safe
df['amount_zscore'] = df['amount_zscore'].fillna(0)

df_sorted = df.sort_values(['AccountID', 'Timestamp'])
df_sorted['time_since_last'] = df_sorted.groupby('AccountID')['Timestamp'].diff().dt.total_seconds()
df['time_since_last'] = df_sorted['time_since_last'].fillna(999999)

df['is_anomaly'] = ((df['amount_zscore'].abs() > 2.0) | (df['time_since_last'] < 300)).astype(int)

target_anomaly_ratio = 0.02
current_anomalies = df['is_anomaly'].sum()
target_anomalies = int(len(df) * target_anomaly_ratio)
needed = max(0, target_anomalies - current_anomalies)

if needed > 0:
    normal_indices = df[df['is_anomaly'] == 0].index.tolist()
    np.random.seed(42)
    new_anomalies = np.random.choice(normal_indices, min(needed, len(normal_indices)), replace=False)

    n1 = needed // 3
    n2 = needed // 3
    n3 = needed - n1 - n2

    df.loc[new_anomalies[:n1], 'TransactionAmount'] *= np.random.uniform(3, 5, n1)
    df.loc[new_anomalies[n1:n1+n2], 'time_since_last'] = np.random.uniform(0, 60, n2)
    df.loc[new_anomalies[n1+n2:], 'AccountBalance'] *= np.random.uniform(0.1, 0.3, n3)
    df.loc[new_anomalies, 'is_anomaly'] = 1

    df['avg_amount_per_account'] = df.groupby('AccountID')['TransactionAmount'].transform('mean')
    df['std_amount_per_account'] = df.groupby('AccountID')['TransactionAmount'].transform('std')
    std_safe = df['std_amount_per_account'].replace(0, 1)
    df['amount_zscore'] = (df['TransactionAmount'] - df['avg_amount_per_account']) / std_safe
    df['amount_zscore'] = df['amount_zscore'].fillna(0)

df['balance_ratio'] = df['TransactionAmount'] / (df['AccountBalance'] + 1e-6)
df['is_large'] = (df['TransactionAmount'] > df['TransactionAmount'].quantile(0.95)).astype(int)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['days_since_start'] = (df['Timestamp'] - df['Timestamp'].min()).dt.days

df = df.sort_values(['AccountID', 'Timestamp'])
df['rolling_mean_amount'] = df.groupby('AccountID')['TransactionAmount'].transform(lambda x: x.rolling(5, min_periods=1).mean())
df['rolling_std_amount'] = df.groupby('AccountID')['TransactionAmount'].transform(lambda x: x.rolling(5, min_periods=1).std().fillna(0))

base_features = ['TransactionAmount', 'AccountBalance', 'hour', 'day_of_week', 'weekend', 'amount_zscore']
new_features = ['balance_ratio', 'is_large', 'hour_sin', 'hour_cos', 'days_since_start', 'rolling_mean_amount', 'rolling_std_amount']
all_features = base_features + new_features

X = df[all_features]
y = df['is_anomaly']

if X.isnull().sum().sum() > 0:
    imputer = SimpleImputer(strategy='mean')
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Baseline RF (class_weight=balanced)': RandomForestClassifier(
        n_estimators=50, max_depth=5, min_samples_leaf=2,
        class_weight='balanced', random_state=42
    ),
    'Balanced Random Forest': BalancedRandomForestClassifier(
        n_estimators=100, max_depth=10, sampling_strategy='auto',
        replacement=True, random_state=42
    ),
    'XGBoost (scale_pos_weight)': XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
        random_state=42, eval_metric='logloss', use_label_encoder=False
    ),
    'LightGBM (is_unbalance)': LGBMClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        is_unbalance=True, random_state=42, verbose=-1
    ),
    'CatBoost (auto_class_weights)': CatBoostClassifier(
        iterations=100, depth=6, learning_rate=0.1,
        auto_class_weights='Balanced', random_seed=42, verbose=False
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
    )
}

smote_pipeline = ImbPipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('smote', SMOTE(random_state=42, sampling_strategy=0.1))
])

X_train_smote, y_train_smote = smote_pipeline.fit_resample(X_train, y_train)

models_smote = {
    'Random Forest + SMOTE': RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42
    ),
    'XGBoost + SMOTE': XGBClassifier(
        n_estimators=100, max_depth=6, random_state=42, eval_metric='logloss'
    ),
    'LightGBM + SMOTE': LGBMClassifier(
        n_estimators=100, max_depth=6, random_state=42, verbose=-1
    )
}

results = []

for name, model in models.items():
    try:
        if 'CatBoost' in name:
            model.fit(X_train, y_train, verbose=False)
        else:
            model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        results.append({
            'Model': name,
            'F1-Score': f1,
            'ROC-AUC': roc_auc,
            'Precision': precision,
            'Recall': recall,
            'TP': tp,
            'FP': fp,
            'FN': fn
        })

    except Exception as e:
        print(f'Ошибка: {str(e)}')

for name, model in models_smote.items():
    try:
        model.fit(X_train_smote, y_train_smote)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        results.append({
            'Model': name,
            'F1-Score': f1,
            'ROC-AUC': roc_auc,
            'Precision': precision,
            'Recall': recall,
            'TP': tp,
            'FP': fp,
            'FN': fn
        })

    except Exception as e:
        print(f'Ошибка: {str(e)}')

results_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False)
print(results_df[['Model', 'F1-Score', 'ROC-AUC', 'Precision', 'Recall', 'TP', 'FP', 'FN']].to_string(index=False))

best_model_name = results_df.iloc[0]['Model']
best_f1 = results_df.iloc[0]['F1-Score']
print(f'Лучшая модель: {best_model_name}')
print(f'F1-Score: {best_f1:.4f}')

                              Model  F1-Score  ROC-AUC  Precision   Recall  TP   FP  FN
              Random Forest + SMOTE  0.492462 0.785162   1.000000 0.326667  49    0 101
                    XGBoost + SMOTE  0.435233 0.731071   0.976744 0.280000  42    1 108
                  Gradient Boosting  0.429825 0.783340   0.628205 0.326667  49   29 101
                   LightGBM + SMOTE  0.404255 0.764239   1.000000 0.253333  38    0 112
            LightGBM (is_unbalance)  0.170704 0.766744   0.103208 0.493333  74  643  76
         XGBoost (scale_pos_weight)  0.147541 0.743646   0.089489 0.420000  63  641  87
      CatBoost (auto_class_weights)  0.126467 0.780695   0.070087 0.646667  97 1287  53
Baseline RF (class_weight=balanced)  0.119975 0.782586   0.066121 0.646667  97 1370  53
             Balanced Random Forest  0.117911 0.778527   0.064378 0.700000 105 1526  45
Лучшая модель: Random Forest + SMOTE
F1-Score: 0.4925


Результаты показали ледующие изменения:
1) Gradient Boosting показал лучший баланс precision/recall (0.63/0.33)
2) Random Forest + SMOTE дал идеальную precision (1.0), но низкий recall (0.33)
3) Baseline модели имеют высокий recall (0.65-0.70), но очень низкую precision (0.06-0.07)

***Вывод***: все модели находят только 30-70% аномалий (recall 0.33-0.70). Для дальнейшей работы лучше всего использовать Gradient Boosting или Random Forest + SMOTE.

In [17]:
best_model = None

if 'Random Forest + SMOTE' in best_model_name:
    best_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    best_model.fit(X_train_smote, y_train_smote)
elif 'XGBoost + SMOTE' in best_model_name:
    best_model = XGBClassifier(n_estimators=100, max_depth=6, random_state=42)
    best_model.fit(X_train_smote, y_train_smote)
elif 'LightGBM + SMOTE' in best_model_name:
    best_model = LGBMClassifier(n_estimators=100, max_depth=6, random_state=42)
    best_model.fit(X_train_smote, y_train_smote)
elif 'Balanced Random Forest' in best_model_name:
    best_model = BalancedRandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    best_model.fit(X_train, y_train)
elif 'XGBoost' in best_model_name:
    best_model = XGBClassifier(n_estimators=100, max_depth=6,
                               scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
                               random_state=42)
    best_model.fit(X_train, y_train)
elif 'CatBoost' in best_model_name:
    best_model = CatBoostClassifier(iterations=100, auto_class_weights='Balanced', random_seed=42, verbose=False)
    best_model.fit(X_train, y_train)
elif 'LightGBM' in best_model_name:
    best_model = LGBMClassifier(n_estimators=100, is_unbalance=True, random_state=42)
    best_model.fit(X_train, y_train)
else:
    best_model = BalancedRandomForestClassifier(n_estimators=100, random_state=42)
    best_model.fit(X_train, y_train)

if best_model:
    joblib.dump(best_model, 'best_model_improved.pkl')
    joblib.dump(scaler, 'scaler_improved.pkl')